# GazeToolkit vs. Tobii Pro Lab — Evaluation

Evaluates [GazeToolkit (Konopka, 2019)](https://github.com/uxifiit/GazeToolkit) against Tobii Pro Lab ground truth on one eye-tracking recording.

## Setup
Both systems implement the **I-VT (Velocity-Threshold) fixation filter** with the same core settings:
- Eye selection: **both-eye average**
- Velocity threshold: **30 °/s**
- Velocity window: **20 ms**

## Label alignment
GazeToolkit uses three output labels: `Fixation`, `Saccade`, `Unknown`.  
Tobii Pro Lab uses four: `Fixation`, `Saccade`, `Unclassified`, `EyesNotFound`.

To ensure a symmetric comparison, the **ground truth is collapsed** to match GazeToolkit's coarser schema:

| Tobii Pro Lab (GT) | → Collapsed GT |
|--------------------|----------------|
| Fixation | Fixation |
| Saccade | Saccade |
| Unclassified | **Unknown** |
| EyesNotFound | **Unknown** |

GazeToolkit's `Unknown` output is kept as-is — no post-hoc patching.

## Two variants
| Variant | GazeToolkit source | GT source |
|---------|-------------------|-----------|
| **No gap fill** | Pre-computed C# pipeline output | No-gap-fill Tobii export |
| **Gap fill 75 ms** | `ivt_filter.py` (Python) on Tobii-gap-filled data | Gap-fill Tobii export |

> **Limitation (gap-fill variant):** `ivt_filter.py` does not implement gap fill itself. It runs on gaze data that Tobii already gap-filled during export. The comparison is therefore *GazeToolkit on Tobii-gap-filled data* vs *Tobii Pro Lab with gap fill 75 ms*, not *GazeToolkit with gap fill*.

---
## 1 — Imports & Paths

In [1]:
import importlib.util
import json
import math
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

# Make the repo root importable (works whether the notebook is run from
# notebooks/ or from the repo root)
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ivt_filter.evaluation.evaluation import compute_ivt_metrics
from ivt_filter.evaluation.event_iou import compute_event_iou_metrics

print("Imports OK")

Imports OK


In [2]:
GAZETOOLKIT_DIR = Path("/home/cem/Documents/Gitprojekt/GazeToolkit")
RESULTS_DIR     = REPO_ROOT / "results" / "gazetoolkit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- no-gap-fill variant ---
NGF_INPUT  = GAZETOOLKIT_DIR / "I-VT-botheye20ms30threshold_input.csv"
NGF_OUTPUT = GAZETOOLKIT_DIR / "I-VT-botheye20ms30threshold_output.tsv"

# --- gap-fill variant ---
GF_TOBII_TSV = (
    GAZETOOLKIT_DIR / "dataset"
    / "IVT-Interpolation75ms-Eyeboth-NoNoise-VelocityWindow20-VelocityTreshold30.tsv"
)

# --- output files ---
RESULTS_JSON       = RESULTS_DIR / "gazetoolkit_eval_results.json"
RAW_NGF_OUT        = RESULTS_DIR / "gazetoolkit_raw_output_no_gap_fill.tsv"
RAW_GF_OUT         = RESULTS_DIR / "gazetoolkit_raw_output_gap_fill.tsv"
SUMMARY_TXT        = RESULTS_DIR / "gazetoolkit_summary.txt"

# Tobii raw-export column → slim name
TOBII_COL_MAP: Dict[str, str] = {
    "Gaze point left X [DACS mm]"  : "gaze_left_x_mm",
    "Gaze point left Y [DACS mm]"  : "gaze_left_y_mm",
    "Gaze point right X [DACS mm]" : "gaze_right_x_mm",
    "Gaze point right Y [DACS mm]" : "gaze_right_y_mm",
    "Gaze point left X [DACS px]"  : "gaze_left_x_px",
    "Gaze point left Y [DACS px]"  : "gaze_left_y_px",
    "Gaze point right X [DACS px]" : "gaze_right_x_px",
    "Gaze point right Y [DACS px]" : "gaze_right_y_px",
    "Validity left"                : "validity_left",
    "Validity right"               : "validity_right",
    "Eye position left X [DACS mm]": "eye_left_x_mm",
    "Eye position left Y [DACS mm]": "eye_left_y_mm",
    "Eye position left Z [DACS mm]": "eye_left_z_mm",
    "Eye position right X [DACS mm]": "eye_right_x_mm",
    "Eye position right Y [DACS mm]": "eye_right_y_mm",
    "Eye position right Z [DACS mm]": "eye_right_z_mm",
    "Eye movement type"            : "gt_event_type",
    "Eye movement type index"      : "gt_event_index",
}

# Timestamp columns to try in order (column name → scale to ms)
TIMESTAMP_FALLBACKS = [
    ("Recording timestamp [ms]", 1.0),
    ("Recording timestamp [μs]", 1e-3),
    ("Eyetracker timestamp [μs]", 1e-3),
]

print("Paths OK")

Paths OK


---
## 2 — Helper Functions

In [3]:
# ── label schemas ─────────────────────────────────────────────────────────────

# GazeToolkit output → evaluation label (Unknown stays Unknown)
GT_LABEL_MAP = {"Fixation": "Fixation", "Saccade": "Saccade", "Unknown": "Unknown"}

def collapse_gt(series: pd.Series) -> pd.Series:
    """
    Collapse Tobii Pro Lab ground truth to GazeToolkit's 3-class schema:
        EyesNotFound  →  Unknown
        Unclassified  →  Unknown
        Fixation      →  Fixation  (unchanged)
        Saccade       →  Saccade   (unchanged)
    """
    mapping = {"EyesNotFound": "Unknown", "Unclassified": "Unknown"}
    return series.fillna("Unknown").astype(str).replace(mapping)


# ── data loading ──────────────────────────────────────────────────────────────

def load_gazetoolkit_input(path: Path) -> pd.DataFrame:
    """Load the slim GazeToolkit input CSV (auto-detect separator)."""
    df = pd.read_csv(path, sep=None, engine="python")
    assert "time_ms" in df.columns
    assert "gt_event_type" in df.columns
    return df.sort_values("time_ms").reset_index(drop=True)


def load_gazetoolkit_events(path: Path) -> pd.DataFrame:
    """Load GazeToolkit event-level output; normalise column names."""
    df = pd.read_csv(path, sep=None, engine="python")
    # ivt_filter.py uses start_time_ms / end_time_ms; C# output uses start_time / end_time
    renames = {}
    if "start_time_ms" in df.columns:
        renames["start_time_ms"] = "start_time"
    if "end_time_ms" in df.columns:
        renames["end_time_ms"] = "end_time"
    if renames:
        df = df.rename(columns=renames)
    assert {"movement_type", "start_time", "end_time"}.issubset(df.columns), \
        f"Missing columns in {path.name}: {list(df.columns)}"
    return df.sort_values("start_time").reset_index(drop=True)


def extract_tobii_input(tobii_tsv: Path) -> pd.DataFrame:
    """Extract slim input format from a 92-column Tobii Pro Lab export."""
    df_raw = pd.read_csv(tobii_tsv, sep="\t", decimal=",", low_memory=False)

    # resolve timestamp column
    ts_col, ts_scale = None, 1.0
    for col, scale in TIMESTAMP_FALLBACKS:
        if col in df_raw.columns:
            ts_col, ts_scale = col, scale
            break
    if ts_col is None:
        raise ValueError("No timestamp column found in Tobii export.")
    print(f"  Timestamp: '{ts_col}' × {ts_scale}")

    available = {k: v for k, v in TOBII_COL_MAP.items() if k in df_raw.columns}
    available[ts_col] = "time_ms"
    df_slim = df_raw[list(available.keys())].rename(columns=available)

    if ts_scale != 1.0:
        df_slim["time_ms"] = df_slim["time_ms"] * ts_scale

    return df_slim.sort_values("time_ms").reset_index(drop=True)


# ── timestamp alignment ────────────────────────────────────────────────────────

def assign_gazetoolkit_labels(samples_df: pd.DataFrame,
                               events_df: pd.DataFrame) -> pd.Series:
    """
    For each sample (time_ms), find the GazeToolkit event where
    start_time ≤ time_ms ≤ end_time.  Samples not covered by any event
    receive 'Unknown'.  Uses merge_asof — O(n log n).
    """
    samples = samples_df[["time_ms"]].copy().sort_values("time_ms")
    events  = (
        events_df[["start_time", "end_time", "movement_type"]]
        .sort_values("start_time")
        .reset_index(drop=True)
    )

    merged = pd.merge_asof(
        samples, events,
        left_on="time_ms", right_on="start_time",
        direction="backward",
    )

    not_covered = merged["end_time"].isna() | (merged["time_ms"] > merged["end_time"])
    merged.loc[not_covered, "movement_type"] = "Unknown"

    labels = merged["movement_type"].map(GT_LABEL_MAP).fillna("Unknown")
    labels.index = samples.index
    return labels.reindex(samples_df.index)


print("Helper functions defined")

Helper functions defined


In [4]:
# ── metric helpers ────────────────────────────────────────────────────────────

def _json_safe(val: Any) -> Any:
    if isinstance(val, float) and math.isnan(val):
        return None
    if isinstance(val, (list, tuple)):
        return [_json_safe(v) for v in val]
    if isinstance(val, dict):
        return {k: _json_safe(v) for k, v in val.items()}
    return val


def _prf(tp: int, n_gt: int, n_pred: int) -> Dict[str, Any]:
    rec  = tp / n_gt   if n_gt   > 0 else float("nan")
    prec = tp / n_pred if n_pred > 0 else float("nan")
    denom = rec + prec
    f1 = (2 * prec * rec / denom
          if not math.isnan(prec) and not math.isnan(rec) and denom > 0
          else float("nan"))
    return dict(
        precision = round(prec * 100, 1) if not math.isnan(prec) else None,
        recall    = round(rec  * 100, 1) if not math.isnan(rec)  else None,
        f1        = round(f1   * 100, 1) if not math.isnan(f1)   else None,
    )


def extract_prf_from_confusion(m: Dict[str, Any]) -> Dict[str, Dict]:
    """Compute per-class precision, recall, F1 from compute_ivt_metrics output."""
    labels: List[str] = m["labels"]
    conf:   List[List[int]] = m["confusion_matrix"]
    k = len(labels)
    result = {}
    for cls in ["Fixation", "Saccade"]:
        if cls not in labels:
            result[cls] = dict(precision=None, recall=None, f1=None)
            continue
        idx    = labels.index(cls)
        tp     = conf[idx][idx]
        n_gt   = sum(conf[idx])
        n_pred = sum(conf[i][idx] for i in range(k))
        result[cls] = _prf(tp, n_gt, n_pred)
    return result


def extract_event_prf(m: Dict[str, Any]) -> Dict[str, Dict]:
    """Compute per-class P/R/F1 from compute_event_iou_metrics output."""
    cm     = m.get("confusion_matrix", {})
    counts = m.get("event_counts", {})
    result = {}
    for cls in ["Fixation", "Saccade"]:
        gt_row = cm.get(cls, {})
        tp     = gt_row.get(cls, 0)
        fn     = gt_row.get("FN", 0)
        wrong  = sum(v for tag, v in gt_row.items() if tag not in (cls, "FN"))
        n_gt   = tp + fn + wrong
        n_pred = counts.get(cls, {}).get("pred", 0)
        d = _prf(tp, n_gt, n_pred)
        d["n_gt"] = n_gt
        d["n_pred"] = n_pred
        result[cls] = d
    return result


def display_results(sample_m: Dict, event_m: Dict, variant_name: str) -> None:
    """Pretty-print evaluation results for one variant."""
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)

    print(f"\n{'='*55}")
    print(f"  {variant_name}")
    print(f"{'='*55}")

    print("\nSample-level")
    print(f"  Agreement (GT Fix/Sac only) : {sample_m['percentage_agreement']:.1f} %")
    print(f"  Agreement (all samples)     : {sample_m['percentage_agreement_all']:.1f} %")
    print(f"  Cohen's κ                   : {sample_m['cohen_kappa']:.3f}")
    print(f"")
    hdr = f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}"
    print(hdr); print("  " + "-"*42)
    for cls in ["Fixation", "Saccade"]:
        p = prf_s[cls]
        print(f"  {cls:<12} {str(p['precision']):>10} {str(p['recall']):>10} {str(p['f1']):>10}")

    print("\nEvent-level  (max-IoU matching, min IoU > 0)")
    hdr2 = f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}  {'GT ev':>6}  {'Pred ev':>7}"
    print(hdr2); print("  " + "-"*56)
    for cls in ["Fixation", "Saccade"]:
        p = prf_e[cls]
        print(f"  {cls:<12} {str(p['precision']):>10} {str(p['recall']):>10} "
              f"{str(p['f1']):>10}  {p['n_gt']:>6}  {p['n_pred']:>7}")


print("Metric helpers defined")

Metric helpers defined


In [5]:
# ── GazeToolkit Python runner ─────────────────────────────────────────────────

def _load_gazetoolkit_module():
    """Import GazeToolkit's ivt_filter.py without name collision."""
    spec = importlib.util.spec_from_file_location(
        "gazetoolkit_ivt_filter",
        str(GAZETOOLKIT_DIR / "ivt_filter.py"),
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def run_gazetoolkit_python(input_df: pd.DataFrame,
                            output_path: Path) -> pd.DataFrame:
    """
    Run GazeToolkit's ivt_filter.py on input_df; save raw TSV to output_path.

    Note: ivt_filter.py uses a fixed viewing distance of 600 mm for angular
    velocity computation (actual distance in this dataset ≈ 565 mm, ~6% off).
    This is a known limitation of the Python implementation.

    Runtime: several minutes (O(n²) velocity loop).
    """
    gtmod = _load_gazetoolkit_module()
    fd, tmp_name = tempfile.mkstemp(suffix=".tsv")
    tmp_path = Path(tmp_name)
    try:
        os.close(fd)
        # Write with comma decimal so ivt_filter.py's decimal=',' parser works
        input_df.to_csv(tmp_path, sep="\t", index=False, decimal=",")
        print("Running ivt_filter.py  (threshold=30°/s, window=20 ms, eye=average)")
        print("This may take several minutes …")
        gtmod.run_ivt_filter(
            str(tmp_path), str(output_path),
            threshold=30.0, window=20.0, eye="average",
        )
    finally:
        tmp_path.unlink(missing_ok=True)

    return load_gazetoolkit_events(output_path)


print("Runner defined")

Runner defined


---
## 3 — Variant 1: No Gap Fill

Uses the **pre-computed C# pipeline output** from GazeToolkit.  
Ground truth from the matching Tobii Pro Lab export (no gap fill).

In [6]:
# Load input (ground truth + timestamps)
ngf_samples = load_gazetoolkit_input(NGF_INPUT)
print(f"Samples : {len(ngf_samples):,}  "
      f"({ngf_samples['time_ms'].min():.0f} – {ngf_samples['time_ms'].max():.0f} ms)")
print("GT label distribution (original):")
print(ngf_samples["gt_event_type"].value_counts().to_string())

Samples : 27,801  (114 – 231782 ms)
GT label distribution (original):
gt_event_type
Fixation        20174
EyesNotFound     4132
Saccade          2805
Unclassified      690


In [7]:
# Load pre-computed GazeToolkit C# output
ngf_events = load_gazetoolkit_events(NGF_OUTPUT)
shutil.copy(NGF_OUTPUT, RAW_NGF_OUT)

print(f"Events  : {len(ngf_events):,}")
print("Event type distribution:")
print(ngf_events["movement_type"].value_counts().to_string())

Events  : 2,516
Event type distribution:
movement_type
Fixation    1258
Saccade      779
Unknown      479


In [8]:
# Collapse GT labels + assign GazeToolkit predictions
ngf_eval = ngf_samples.copy()
ngf_eval["gt_collapsed"]       = collapse_gt(ngf_eval["gt_event_type"])
ngf_eval["gazetoolkit_label"]  = assign_gazetoolkit_labels(ngf_eval, ngf_events)

print("Collapsed GT distribution:")
print(ngf_eval["gt_collapsed"].value_counts().to_string())
print("\nGazeToolkit prediction distribution:")
print(ngf_eval["gazetoolkit_label"].value_counts().to_string())

Collapsed GT distribution:
gt_collapsed
Fixation    20174
Unknown      4822
Saccade      2805

GazeToolkit prediction distribution:
gazetoolkit_label
Fixation    20891
Unknown      4133
Saccade      2777


In [9]:
# Cross-tabulation (sanity check)
pd.crosstab(
    ngf_eval["gt_collapsed"],
    ngf_eval["gazetoolkit_label"],
    margins=True,
)

gazetoolkit_label,Fixation,Saccade,Unknown,All
gt_collapsed,,,,
Fixation,20075,98,1,20174
Saccade,126,2679,0,2805
Unknown,690,0,4132,4822
All,20891,2777,4133,27801


In [10]:
# Sample-level metrics
ngf_sample_m = compute_ivt_metrics(
    ngf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
)

# Event-level metrics (max-IoU)
ngf_event_m = compute_event_iou_metrics(
    ngf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
    time_col="time_ms",
    event_types=["Fixation", "Saccade"],
)

display_results(ngf_sample_m, ngf_event_m, "Variant 1 — No Gap Fill  (C# pipeline)")


  Variant 1 — No Gap Fill  (C# pipeline)

Sample-level
  Agreement (GT Fix/Sac only) : 99.0 %
  Agreement (all samples)     : 96.7 %
  Cohen's κ                   : 0.921

  Class         Precision     Recall         F1
  ------------------------------------------
  Fixation           96.1       99.5       97.8
  Saccade            96.5       95.5       96.0

Event-level  (max-IoU matching, min IoU > 0)
  Class         Precision     Recall         F1   GT ev  Pred ev
  --------------------------------------------------------
  Fixation           57.6       98.1       72.5     738     1258
  Saccade            88.6       94.0       91.2     734      779


### Notes — No Gap Fill

- **High sample-level F1** for both classes, but **low event-level Fixation precision**: GazeToolkit produces ~1 258 fixation events vs. ~738 in the GT. This indicates the C# pipeline output was generated **without a fixation-merge step** — long fixations are fragmented into many short consecutive events. The sample-level label is still mostly correct (a fragment still gets labelled Fixation), hence the disconnect between sample and event agreement.
- The **C# pipeline generation settings are not documented** (no record of whether merge/discard steps were applied).

---
## 4 — Variant 2: Gap Fill 75 ms

Runs **GazeToolkit's Python implementation** (`ivt_filter.py`) on the Tobii-gap-filled gaze export.

> ⚠️ `ivt_filter.py` does **not** implement gap fill. It sees Tobii's already-interpolated gaze data as input. The comparison is *GazeToolkit on Tobii-gap-filled data* vs. *Tobii Pro Lab classification with gap fill 75 ms*.

In [11]:
# Extract slim input from the 92-column Tobii export
print(f"Loading: {GF_TOBII_TSV.name}")
gf_samples = extract_tobii_input(GF_TOBII_TSV)
print(f"Samples : {len(gf_samples):,}  "
      f"({gf_samples['time_ms'].min():.0f} – {gf_samples['time_ms'].max():.0f} ms)")
print("GT label distribution (original):")
print(gf_samples["gt_event_type"].value_counts().to_string())

Loading: IVT-Interpolation75ms-Eyeboth-NoNoise-VelocityWindow20-VelocityTreshold30.tsv


  Timestamp: 'Recording timestamp [μs]' × 0.001
Samples : 27,822  (0 – 231939 ms)
GT label distribution (original):
gt_event_type
Fixation        20991
EyesNotFound     3600
Saccade          3226
Unclassified        3


In [12]:
# Load pre-computed GazeToolkit output (or re-run ivt_filter.py if absent)
if RAW_GF_OUT.exists():
    print(f"Loading pre-computed output: {RAW_GF_OUT.name}")
    gf_events = load_gazetoolkit_events(RAW_GF_OUT)
else:
    # ivt_filter.py has an O(n²) velocity loop - may take several minutes
    gf_events = run_gazetoolkit_python(gf_samples, RAW_GF_OUT)

print("")
print(f"Events  : {len(gf_events):,}")
print("Event type distribution:")
print(gf_events["movement_type"].value_counts().to_string())


Loading pre-computed output: gazetoolkit_raw_output_gap_fill.tsv

Events  : 1,796
Event type distribution:
movement_type
Saccade     803
Fixation    713
Unknown     280


In [13]:
# Collapse GT labels + assign GazeToolkit predictions
gf_eval = gf_samples.copy()
gf_eval["gt_collapsed"]      = collapse_gt(gf_eval["gt_event_type"])
gf_eval["gazetoolkit_label"] = assign_gazetoolkit_labels(gf_eval, gf_events)

print("Collapsed GT distribution:")
print(gf_eval["gt_collapsed"].value_counts().to_string())
print("\nGazeToolkit prediction distribution:")
print(gf_eval["gazetoolkit_label"].value_counts().to_string())

Collapsed GT distribution:


gt_collapsed
Fixation    20991
Unknown      3605
Saccade      3226

GazeToolkit prediction distribution:
gazetoolkit_label
Fixation    18751
Unknown      6439
Saccade      2632


In [14]:
# Cross-tabulation
pd.crosstab(
    gf_eval["gt_collapsed"],
    gf_eval["gazetoolkit_label"],
    margins=True,
)

gazetoolkit_label,Fixation,Saccade,Unknown,All
gt_collapsed,,,,
Fixation,18715,173,2103,20991
Saccade,36,2459,731,3226
Unknown,0,0,3605,3605
All,18751,2632,6439,27822


In [15]:
# Metrics
gf_sample_m = compute_ivt_metrics(
    gf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
)

gf_event_m = compute_event_iou_metrics(
    gf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
    time_col="time_ms",
    event_types=["Fixation", "Saccade"],
)

display_results(gf_sample_m, gf_event_m, "Variant 2 — Gap Fill 75 ms  (ivt_filter.py)")


  Variant 2 — Gap Fill 75 ms  (ivt_filter.py)

Sample-level
  Agreement (GT Fix/Sac only) : 87.4 %
  Agreement (all samples)     : 89.1 %
  Cohen's κ                   : 0.757

  Class         Precision     Recall         F1
  ------------------------------------------
  Fixation           99.8       89.2       94.2
  Saccade            93.4       76.2       84.0

Event-level  (max-IoU matching, min IoU > 0)
  Class         Precision     Recall         F1   GT ev  Pred ev
  --------------------------------------------------------
  Fixation           91.1       75.6       82.7     857      711
  Saccade            81.7       74.7       78.0     866      792


### Notes — Gap Fill 75 ms

- **Lower agreement** than Variant 1, partly because Tobii's gap fill bridges periods where both eyes are invalid and classifies those samples as Fixation/Saccade, while GazeToolkit (without gap fill) labels them Unknown.
- `ivt_filter.py` uses a **fixed viewing distance of 600 mm** for angular velocity; the actual distance in this dataset is ≈ 565 mm (~6% underestimation of velocity).
- The Python implementation uses `eye='average'` with a strict **both-eyes-valid** requirement. Samples where only one eye is valid are treated as Unknown; Tobii can still classify these.

---
## 5 — Side-by-Side Comparison

In [16]:
def make_summary_df(sample_m, event_m, name):
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)
    rows = {
        "Variant": name,
        "Agreement Fix/Sac (%)": round(sample_m["percentage_agreement"], 1),
        "Agreement all (%)": round(sample_m["percentage_agreement_all"], 1),
        "Cohen's κ": round(sample_m["cohen_kappa"], 3),
        "Fix Precision (%)": prf_s["Fixation"]["precision"],
        "Fix Recall (%)": prf_s["Fixation"]["recall"],
        "Fix F1 (%)": prf_s["Fixation"]["f1"],
        "Sac Precision (%)": prf_s["Saccade"]["precision"],
        "Sac Recall (%)": prf_s["Saccade"]["recall"],
        "Sac F1 (%)": prf_s["Saccade"]["f1"],
        "Fix F1 event (%)": prf_e["Fixation"]["f1"],
        "Sac F1 event (%)": prf_e["Saccade"]["f1"],
        "GT Fix events": prf_e["Fixation"]["n_gt"],
        "Pred Fix events": prf_e["Fixation"]["n_pred"],
        "GT Sac events": prf_e["Saccade"]["n_gt"],
        "Pred Sac events": prf_e["Saccade"]["n_pred"],
    }
    return rows


comparison = pd.DataFrame([
    make_summary_df(ngf_sample_m, ngf_event_m, "No Gap Fill (C#)"),
    make_summary_df(gf_sample_m,  gf_event_m,  "Gap Fill 75ms (Python)"),
]).set_index("Variant")

comparison

,Agreement Fix/Sac (%),Agreement all (%),Cohen's κ,Fix Precision (%),Fix Recall (%),Fix F1 (%),Sac Precision (%),Sac Recall (%),Sac F1 (%),Fix F1 event (%),Sac F1 event (%),GT Fix events,Pred Fix events,GT Sac events,Pred Sac events
Variant,,,,,,,,,,,,,,,
No Gap Fill (C#),99.0,96.7,0.921,96.1,99.5,97.8,96.5,95.5,96.0,72.5,91.2,738,1258,734,779
Gap Fill 75ms (Python),87.4,89.1,0.757,99.8,89.2,94.2,93.4,76.2,84.0,82.7,78.0,857,711,866,792


---
## 6 — Save Outputs

In [17]:
def serialise_metrics(sample_m, event_m):
    """Make both metric dicts JSON-safe (drop MatchResult lists, handle NaN)."""
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)
    sl = _json_safe({k: v for k, v in sample_m.items() if k != "confusion_matrix"})
    sl["confusion_matrix"] = _json_safe(sample_m["confusion_matrix"])
    sl["confusion_matrix_labels"] = sample_m["labels"]
    sl["per_class"] = prf_s
    el = _json_safe({k: v for k, v in event_m.items()
                     if k not in ("matches", "unmatched_pred")})
    el["per_class"] = prf_e
    return {"sample_level": sl, "event_level": el}


results = {
    "no_gap_fill": serialise_metrics(ngf_sample_m, ngf_event_m),
    "gap_fill":    serialise_metrics(gf_sample_m,  gf_event_m),
    "label_schema": {
        "gt_collapse": {
            "EyesNotFound": "Unknown",
            "Unclassified": "Unknown",
        },
        "classes": ["Fixation", "Saccade", "Unknown"],
    },
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"JSON saved  → {RESULTS_JSON}")
print(f"Raw NGF TSV → {RAW_NGF_OUT}")
print(f"Raw GF TSV  → {RAW_GF_OUT}")

JSON saved  → /home/cem/Documents/Gitprojekt/Tobii-I-VT-Filter-Reconstruction/results/gazetoolkit/gazetoolkit_eval_results.json
Raw NGF TSV → /home/cem/Documents/Gitprojekt/Tobii-I-VT-Filter-Reconstruction/results/gazetoolkit/gazetoolkit_raw_output_no_gap_fill.tsv
Raw GF TSV  → /home/cem/Documents/Gitprojekt/Tobii-I-VT-Filter-Reconstruction/results/gazetoolkit/gazetoolkit_raw_output_gap_fill.tsv


In [18]:
# Plain-text summary for thesis
def _f(v, d=1):
    return "N/A" if v is None else f"{v:.{d}f}"

lines = [
    "GazeToolkit vs Tobii Pro Lab — Evaluation Summary",
    "=" * 55,
    "",
    "Settings : both-eye average | threshold 30 °/s | window 20 ms",
    "Labels   : Fixation / Saccade / Unknown",
    "           (Tobii EyesNotFound + Unclassified collapsed → Unknown)",
    "",
]

for variant_key, sm, em, title in [
    ("no_gap_fill", ngf_sample_m, ngf_event_m, "Variant 1 — No Gap Fill  (C# pipeline)"),
    ("gap_fill",    gf_sample_m,  gf_event_m,  "Variant 2 — Gap Fill 75 ms  (ivt_filter.py)"),
]:
    prf_s = extract_prf_from_confusion(sm)
    prf_e = extract_event_prf(em)
    lines += [
        title, "-" * len(title), "",
        "Sample-level",
        f"  Agreement (Fix/Sac only) : {_f(sm['percentage_agreement'])} %",
        f"  Agreement (all samples)  : {_f(sm['percentage_agreement_all'])} %",
        f"  Cohen's κ                : {_f(sm['cohen_kappa'], 3)}",
        f"",
        f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}",
        f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}",
    ]
    for cls in ["Fixation", "Saccade"]:
        p = prf_s[cls]
        lines.append(f"  {cls:<12} {_f(p['precision']):>10} "
                     f"{_f(p['recall']):>10} {_f(p['f1']):>10}")
    lines += [
        "",
        "Event-level  (max-IoU, min IoU > 0)",
        f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}  GT ev  Pred ev",
        f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}  -----  -------",
    ]
    for cls in ["Fixation", "Saccade"]:
        p = prf_e[cls]
        lines.append(f"  {cls:<12} {_f(p['precision']):>10} "
                     f"{_f(p['recall']):>10} {_f(p['f1']):>10}  "
                     f"{p['n_gt']:>5}  {p['n_pred']:>7}")
    lines.append("")

lines += [
    "Limitations",
    "-----------",
    "No gap fill: C# pipeline generation settings unknown (no merge step evident).",
    "Gap fill   : ivt_filter.py uses fixed 600 mm viewing distance (actual ≈565 mm);",
    "             no native gap fill; strict both-eyes-valid requirement.",
]

summary_text = "\n".join(lines)

with open(SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)

GazeToolkit vs Tobii Pro Lab — Evaluation Summary

Settings : both-eye average | threshold 30 °/s | window 20 ms
Labels   : Fixation / Saccade / Unknown
           (Tobii EyesNotFound + Unclassified collapsed → Unknown)

Variant 1 — No Gap Fill  (C# pipeline)
--------------------------------------

Sample-level
  Agreement (Fix/Sac only) : 99.0 %
  Agreement (all samples)  : 96.7 %
  Cohen's κ                : 0.921

  Class         Precision     Recall         F1
  ------------ ---------- ---------- ----------
  Fixation           96.1       99.5       97.8
  Saccade            96.5       95.5       96.0

Event-level  (max-IoU, min IoU > 0)
  Class         Precision     Recall         F1  GT ev  Pred ev
  ------------ ---------- ---------- ----------  -----  -------
  Fixation           57.6       98.1       72.5    738     1258
  Saccade            88.6       94.0       91.2    734      779

Variant 2 — Gap Fill 75 ms  (ivt_filter.py)
-------------------------------------------

Samp